# Visualization Capstone

**REQUIRED DAY 3**

## The most important lesson of the day

Lessons 04-08 produced four raw results tables: a differential expression table, a proportions table, an enrichment table, a program-usage matrix. A table is not a communication — almost nobody outside this room will read 10,086 rows of DE statistics. This notebook turns those four tables into four figures someone else could actually understand.

Every choice below — chart type, color, what gets labeled — follows a deliberate method, not default styling. Four non-negotiables show up in all four figures: **one hue for magnitude** (sequential), **two hues + neutral for direction** (diverging), **the same color always means the same condition**, and **label only what matters**, not every point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# Colors chosen deliberately, not matplotlib defaults -- see markdown below each figure for why.
BLUE = "#2a78d6"     # categorical slot 1 -- ctrl
ORANGE = "#eb6834"   # categorical slot 2 -- stim
RED = "#e34948"      # diverging pole -- up
GRAY = "#c3c2b7"     # muted / not-significant / connecting lines

## Figure 1: the DE volcano (lesson 04)

**Form**: a scatter plot is the right form here because the job is showing *every* gene's magnitude (log2 fold change) against its significance (-log10 padj) at once — not a single summary number. **Color**: this is a **diverging** encoding (direction of change, with a real neutral: not significant) — blue/red, never a rainbow, and gray (not a third saturated hue) for the genes that don't matter to the story. **Labels**: only the 6 most significant genes get names — labeling all 10,086 points would be unreadable; the point of selective labeling is to name the genes the reader should recognize (ISG15-family genes), not to caption every dot.

In [ ]:
pb = pd.read_csv("results/de_results_cd14mono.csv", index_col=0).dropna(subset=["padj"])

pb["neglog10padj"] = -np.log10(pb["padj"].clip(lower=1e-300))
sig_up = (pb["padj"] < 0.05) & (pb["log2FoldChange"] > 1)
sig_down = (pb["padj"] < 0.05) & (pb["log2FoldChange"] < -1)
notsig = ~(sig_up | sig_down)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pb.loc[notsig, "log2FoldChange"], pb.loc[notsig, "neglog10padj"], s=8, c=GRAY, alpha=0.5, linewidths=0, label="not significant")
ax.scatter(pb.loc[sig_up, "log2FoldChange"], pb.loc[sig_up, "neglog10padj"], s=10, c=RED, alpha=0.7, linewidths=0, label="up in stim")
ax.scatter(pb.loc[sig_down, "log2FoldChange"], pb.loc[sig_down, "neglog10padj"], s=10, c=BLUE, alpha=0.7, linewidths=0, label="down in stim")

for gene, row in pb.sort_values("padj").head(6).iterrows():
    ax.annotate(gene, (row["log2FoldChange"], row["neglog10padj"]), fontsize=8, xytext=(3, 3), textcoords="offset points")

ax.set_xlabel("log2 fold change (stim vs ctrl)")
ax.set_ylabel("-log10(padj)")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)  # recessive chrome -- the data is the point, not the box around it
fig.tight_layout()

## Figure 2: paired composition (lesson 05)

**Form**: since the whole statistical point of lesson 05 was *pairing* (same donor, both conditions), the figure has to show pairing visually, not just plot two clouds of points next to each other. A thin connecting line per donor, per cell type, makes the paired structure impossible to miss — and incidentally makes the honest null result from lesson 05 visible too: most lines are short, meaning most donors barely moved. **Color**: fixed **categorical** assignment — ctrl is always blue, stim is always orange, in every figure in this notebook, so a reader never has to re-learn the legend.

In [ ]:
adata = sc.read_h5ad("/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad")
counts = adata.obs.groupby(["replicate", "label", "cell_type"], observed=True).size().reset_index(name="n")
totals = counts.groupby(["replicate", "label"], observed=True)["n"].transform("sum")
counts["prop"] = counts["n"] / totals
stim = counts[counts["label"] == "stim"].pivot(index="replicate", columns="cell_type", values="prop").fillna(0)
ctrl = counts[counts["label"] == "ctrl"].pivot(index="replicate", columns="cell_type", values="prop").fillna(0)
stim, ctrl = stim.align(ctrl, join="inner")

fig, ax = plt.subplots(figsize=(7, 4))
cell_types = stim.columns.tolist()
x = np.arange(len(cell_types))
JITTER = 0.12
for i, ct in enumerate(cell_types):
    for donor in stim.index:
        ax.plot([x[i] - JITTER, x[i] + JITTER], [ctrl.loc[donor, ct], stim.loc[donor, ct]],
                color=GRAY, lw=1.0, alpha=0.7, zorder=1)
    ax.scatter(np.full(len(ctrl), x[i] - JITTER), ctrl[ct].values, s=25, c=BLUE, alpha=0.9, zorder=2, label="ctrl" if i == 0 else None)
    ax.scatter(np.full(len(stim), x[i] + JITTER), stim[ct].values, s=25, c=ORANGE, alpha=0.9, zorder=2, label="stim" if i == 0 else None)
ax.set_xticks(x)
ax.set_xticklabels(cell_types, rotation=30, ha="right")
ax.set_ylabel("proportion of cells")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## Figure 3: pathway enrichment (lesson 06)

**Form**: a horizontal ranked bar chart — the job here is pure magnitude comparison across a one-dimensional list (which pathway is more enriched), sorted so the eye reads top-to-bottom as most-to-least. **Color**: **sequential**, one hue (blue), light-to-dark — this is magnitude, not direction, so a diverging or categorical palette would be the wrong tool here, not just a style choice.

In [ ]:
import decoupler as dc

hallmark = pd.read_csv("/tscc/nfs/home/juf009/day3_shared_data/hallmark_genesets.csv")
stat_df = pb[["log2FoldChange"]].dropna().T
stat_df.index = ["CD14_Monocytes_stim_vs_ctrl"]
es, pv = dc.mt.ora(stat_df, hallmark, tmin=5)
result = pd.DataFrame({"pathway": es.columns, "score": es.iloc[0].values, "pval": pv.iloc[0].values})
result = result.sort_values("score", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(6, 4))
seq_blues = plt.cm.Blues(np.linspace(0.4, 0.9, len(result)))
ax.barh(result["pathway"][::-1], result["score"][::-1], color=seq_blues[::-1])
ax.set_xlabel("ORA enrichment score")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

## Figure 4: gene expression program usage (lesson 08)

**Form**: a small matrix (7 programs x 2 conditions) is exactly what a heatmap is for — and small enough that direct numeric labels on every cell are the right call here (unlike Figure 1's 10,086 genes, where labeling everything would be unreadable — the same principle, opposite conclusion, because the data size is different). **Color**: sequential blue again, same reasoning as Figure 3 — this is magnitude (how much a program is used), not direction.

In [ ]:
GRID_DIR = "/tscc/nfs/home/juf009/day3_shared_data/cd4_full_grid"
usage = pd.read_csv(f"{GRID_DIR}/cd4_full_grid.usages.k_7.dt_0_5.consensus.txt", sep="\t", index_col=0)
usage_norm = usage.div(usage.sum(axis=1), axis=0)
usage_norm.columns = [f"program_{c}" for c in usage_norm.columns]
sub_obs = adata.obs.loc[adata.obs["cell_type"] == "CD4 T cells", ["label"]]
usage_norm = usage_norm.reindex(sub_obs.index).join(sub_obs)
heat_data = usage_norm.groupby("label", observed=True).mean().T

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(heat_data.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels(heat_data.columns)
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels(heat_data.index)
for i in range(heat_data.shape[0]):
    for j in range(heat_data.shape[1]):
        val = heat_data.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8,
                color="white" if val > heat_data.values.max() / 2 else "black")
fig.colorbar(im, ax=ax, label="mean usage")
fig.tight_layout()

## What NOT to do (the anti-patterns these four figures deliberately avoid)

- **No rainbow colormap** (`jet`, default `viridis`-for-everything) — sequential magnitude gets one hue, not a hue that implies categories where there are none.
- **No dual y-axis** — if two measures need comparing, that's two figures or small multiples, never two scales stacked on one plot pretending to be one chart.
- **No color-only encoding for anything a colorblind reader needs** — every figure above also uses position, shape, or a direct label, not color alone.
- **No labeling every point** — Figure 1 has 10,086 genes and 6 labels; Figure 4 has 14 cells and 14 labels. The right amount of labeling depends on the data size, not a fixed habit.

## Practice: the full 24-item checklist, end to end

This is today's capstone. Open a fresh Agent B session and run the **entire extended 24-item checklist** from [02_agent_assisted_biological_inference_workflow.md](02_agent_assisted_biological_inference_workflow.md) against your complete pipeline — data loading through visualization. Write down, for each item, the PASS/WARNING/FAIL verdict, the evidence, and the smallest correction if it's not a clean PASS.

## Further reading

- [Choosing a chart form](https://www.data-to-viz.com/) — a decision-tree-style guide to matching form to data job.
- [ColorBrewer](https://colorbrewer2.org/) — validated sequential/diverging/categorical palettes, the same principle used for the colors in this notebook.